# 04 · Encoding: sentence embeddings
Step 4 of the thesis workflow (after `03_baselines`).

**Input:** `data/raw/journal_entries_4203.csv`, `data/derived/meta.parquet`, `data/derived/onehot_features.json`
**Output (per template):** `data/derived/emb_<template>.npy` (unique sentences × 384), `emb_<template>_index.npy` (one sentence id per entry, `meta.parquet` order), `emb_<template>.json` (sentences, counts, model, versions)

Decision D16 (`results/feature_engineering_plan.md`):

| Template | Sentence (entry `a_1`) | Role |
|---|---|---|
| `tc` (T-C) | `Debit Deductible input tax, 19%; Debit Cost of merchandise, 19% input tax; Credit Trade payables.` | main setting: embedding + the non-account one-hot columns |
| `bak` (T-Bak) | `Account_DC: 1576_D, Account_DC: 3400_D, Account_DC: 1600_C` | ablation: exact Bakumenko et al. (2025) format with codes |

T-A (text only) uses the `tc` embedding without columns, so it needs no separate file. How the embedding and the columns are weighted (D17) is decided before the week-3 runs; this notebook stores the raw, L2-normalised embeddings only.

**Needs:** `pip install sentence-transformers` (pulls in torch). The model (~90 MB) is downloaded once. Runs in seconds on a laptop CPU, because the whole ledger has only 29 distinct sentences.

In [ ]:
# cell_04_01
import sys
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)
RAW_REL = Path("data/raw/journal_entries_4203.csv")


def find_project_root(marker: Path = RAW_REL) -> Path:
    # walk up from the working directory until `marker` exists (same as notebooks 01-03)
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / marker).exists():
            return p
    raise FileNotFoundError(f"{marker} not found from {here} upwards")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT))
from src.load import load_lines
from src import embed, encode as fe

DATA = ROOT / RAW_REL
DERIVED = ROOT / "data" / "derived"
print("project root:", ROOT.name, "| model:", embed.MODEL_NAME)

## 1. Load lines and the meta table

In [ ]:
# cell_04_02
lines = load_lines(DATA)
meta = pd.read_parquet(DERIVED / "meta.parquet")
print(lines.shape, meta.shape)
# account names are unique per account (25 accounts = 25 names), so names lose no information
assert lines.groupby("gl_account")["gl_account_name"].nunique().max() == 1
print("accounts:", lines["gl_account"].nunique(), "| names:", lines["gl_account_name"].nunique())

## 2. Build the sentences (D16, lines in D9 order)

In [ ]:
# cell_04_03
sentences, texts, index = {}, {}, {}
for t in embed.TEMPLATES:
    sentences[t] = embed.build_sentences(lines, meta, t)
    texts[t], index[t] = embed.unique_index(sentences[t])
    assert sentences[t].notna().all() and len(sentences[t]) == len(meta)
    print(f"{t}: {len(texts[t])} distinct sentences for {len(meta):,} entries")
i_a1 = int(np.flatnonzero(meta["posting_id"].to_numpy() == "a_1")[0])
for t in embed.TEMPLATES:
    print(f"  a_1 [{t}]: {sentences[t].iloc[i_a1]}")

## 3. The distinct booking patterns

Labels are used here only to describe the patterns (no choice is made from them). Only M1 (cash-in-hand instead of trade payables) and M7 (rare bank account 1200 "Bank") change the account pattern; the other types sit inside the frequent normal patterns.

In [ ]:
# cell_04_04
desc = (pd.DataFrame({"sentence": sentences["tc"], "label": meta["label"].to_numpy(),
                      "marking": meta["marking"].to_numpy()})
        .groupby("sentence", sort=False)
        .agg(entries=("label", "size"), anomalies=("label", "sum"),
             types=("marking", lambda s: ",".join(f"M{m}" for m in sorted(set(s)) if m > 0)))
        .sort_values("entries", ascending=False))
desc

## 4. Embed the distinct sentences (all-MiniLM-L6-v2, L2-normalised)

In [ ]:
# cell_04_05
import time
emb = {}
for t in embed.TEMPLATES:
    t0 = time.perf_counter()
    emb[t] = embed.encode(texts[t])
    print(f"{t}: {emb[t].shape} in {time.perf_counter() - t0:.1f} s")
    assert np.allclose(np.linalg.norm(emb[t], axis=1), 1.0, atol=1e-4)

## 5. First look: how close are the patterns that matter?

Cosine similarity between the normal pattern and the two anomaly patterns that change accounts. Hypothesis (D16): names separate M1 well ("Cash-in-hand" vs "Trade payables"), but may hide M7 ("Bank" vs "Bank (Postbank)"); codes may behave differently.

In [ ]:
# cell_04_06
def sim(t, a_tc, b_tc):
    # similarity of two patterns, given by their T-C sentences, under template t
    ia = int(np.flatnonzero(sentences["tc"].to_numpy() == a_tc)[0])
    ib = int(np.flatnonzero(sentences["tc"].to_numpy() == b_tc)[0])
    va, vb = emb[t][index[t][ia]], emb[t][index[t][ib]]
    return float(va @ vb)

pairs = {
    "M7: payment via Bank (1200) vs normal payment via Bank (Postbank)":
        ("Debit Trade payables; Credit Bank (Postbank).", "Debit Trade payables; Credit Bank."),
    "M1: purchase paid from cash-in-hand vs normal purchase on account":
        ("Debit Deductible input tax, 19%; Debit Cost of merchandise, 19% input tax; Credit Trade payables.",
         "Debit Deductible input tax, 19%; Debit Cost of merchandise, 19% input tax; Credit Cash-in-hand."),
    "reference: payment vs receipt (sides swapped)":
        ("Debit Trade payables; Credit Bank (Postbank).", "Debit Bank (Postbank); Credit Trade receivables."),
}
pd.DataFrame({name: {t: sim(t, *p) for t in embed.TEMPLATES} for name, p in pairs.items()}).T.round(3)

## 6. Save (unique embeddings + index + JSON with sentences, model and versions)

In [ ]:
# cell_04_07
import sentence_transformers, torch
info = {"model": embed.MODEL_NAME, "normalised": True, "line_order": "D9",
        "sentence_transformers": sentence_transformers.__version__, "torch": torch.__version__,
        "dim": int(emb["tc"].shape[1]), "n_entries": int(len(meta))}
for t in embed.TEMPLATES:
    embed.save(DERIVED, t, texts[t], index[t], emb[t], info)
    print("saved", DERIVED / f"emb_{t}.npy", DERIVED / f"emb_{t}_index.npy", DERIVED / f"emb_{t}.json")

## 7. How later notebooks use it

In [ ]:
# cell_04_08
E, info_tc = embed.load(DERIVED, "tc")                    # one row per entry, meta order
print("T-C embedding per entry:", E.shape, "| distinct rows:", len(info_tc["texts"]))

X, feature_info, params = fe.load_encoding(DERIVED / "X_onehot.npz", DERIVED / "onehot_features.json")
cols = embed.non_account_columns(feature_info, fe.FEATURE_GROUPS_MAIN)
print("non-account one-hot columns kept next to the embedding (main groups):", len(cols))
print([feature_info[i]["name"] for i in cols])
# The combination [embedding | these columns] and its weighting (D17) are built in week 3.

## Notes for the thesis

- Ch. 3 (encoding): template table above, the three changes to Bakumenko et al. (2025) and why (D16); only 29 distinct sentences, so the embedding is a semantic embedding of the booking pattern; numbers stay as columns (Wallace et al. 2019; Thawani et al. 2021).
- Section 5 gives the first evidence for the M1 / M7 hypothesis; the detector results in week 3 decide.
- Model and library versions are stored in `emb_<template>.json` for reproducibility.